# Notebook 03c: Co-Occurrence Network and Combination Analysis

**Pipeline layer E.** Reads the enriched stockiness table from 03b and rebuilds the per-film element membership from 02. Produces per-genre co-occurrence networks, per-node centrality measures, and per-pair combination-stockiness scores. Non-destructive: the original stockiness tables are not altered.

## Purpose

The scènes à faire doctrine holds that individual stock elements are unprotectable, but a combination of stock elements can be original. The Indiana Jones illustration is the cleanest version of this. A Nazi villain is stock in WWII films; a magical buried artifact is stock in adventure films; a swastika is a stock visual symbol. None of those ingredients are protectable as scènes à faire. But the combination — Nazis seeking a buried magical artifact — was original expression when *Raiders of the Lost Ark* introduced it, because no prior work had united those specific stock ingredients in that specific configuration. The element-level analysis in 03 and 03b scores each ingredient individually. This notebook scores combinations.

The construction is the same per genre and answers two doctrinal questions:

1. **Which elements are constitutive of a genre's vocabulary?** Centrality measures over the co-occurrence graph identify load-bearing conventions. High-eigenvector-centrality elements are not merely frequent in the genre; they are connected to many other frequent elements, so they sit at the genre's structural core. These are the most likely candidates for what case law treats as the unprotectable conventional vocabulary of the genre.

2. **Which pairs of stock elements are themselves stock combinations?** A pair where both ingredients are individually stock in the genre AND the pair co-occurs more than chance is, as a combination, also conventional and unprotectable. A pair where both ingredients are stock but the pair co-occurs at chance rate or rarely is a candidate for original combination: the ingredients are not protectable individually, but the configuration that unites them may carry originality.

## Method

The same construction runs once per canonical genre.

**Nodes.** Every narrative element from 03b (`is_narrative_element == True`) that appears in at least five films of the genre. Node identity is the `(element, element_type)` pair, preserving the type distinction the pipeline has maintained since 02.

**Edges.** Every pair of nodes that co-occur in at least five films of the genre. The five-film floor matches the corpus-wide film threshold in 03 and keeps the graph robust to hapax noise.

**Edge weight.** Within-genre positive pointwise mutual information:

$$\text{PMI}_+(e_i, e_j \mid g) = \max\!\left(0,\ \log \frac{P(e_i, e_j \mid g)}{P(e_i \mid g)\, P(e_j \mid g)}\right)$$

The probabilities are computed at the film level within the genre. Signed PMI is stored separately for transparency, but the positive variant is what weights the network because centrality with negative weights is poorly defined.

**Centrality.** Three measures per node, all from the within-genre PMI-weighted graph:
- Degree centrality (normalized): the count of neighbors divided by $(n-1)$.
- Eigenvector centrality: identifies elements connected to other highly-connected elements.
- PageRank with damping $d = 0.85$: a robustness check on the eigenvector calculation. Substantial disagreement between the two is reported.

Alongside each centrality value the table carries `z_ensemble_in_genre` — the per-element four-measure ensemble z-score in this genre from 03. The reason: raw eigenvector centrality on a co-occurrence graph identifies *structural hubs*, which on most genres turn out to be generic high-frequency elements (`body`, `house`, `room`) that touch nearly every other element. The doctrinally meaningful "load-bearing convention" is the element that is *both* structurally central in the genre's network *and* distinctively belongs to the genre. The report exposes both views: raw eigenvector centrality (structural hubs), and eigenvector centrality among elements with `z_ensemble_in_genre >= 1` (load-bearing conventions).

**Combination stockiness.** Per surviving pair $(e_i, e_j, g)$:

- Joint film count and joint frequency within the genre.
- Pair-level posterior $P(g \mid e_i, e_j)$, Dirichlet-smoothed with $\alpha = 1$ and the corpus film-genre prior used in 03's posterior member of the ensemble.
- Corpus lift, defined as the ratio of the corpus joint probability to the product of corpus marginals:
$$\text{lift}(e_i, e_j) = \frac{P(e_i, e_j \mid \text{corpus})}{P(e_i \mid \text{corpus})\, P(e_j \mid \text{corpus})}$$
which is the exponential of the corpus-wide PMI.
- Composite combination-stockiness score:
$$\text{CS}(e_i, e_j \mid g) = \tfrac{1}{2}\bigl(z_{\text{ens}}(e_i, g) + z_{\text{ens}}(e_j, g)\bigr)\,\log\text{lift}(e_i, e_j)$$
where $z_{\text{ens}}$ is the four-measure ensemble z-score from 03. The composite is high when both ingredients are stock in $g$ AND they co-occur super-additively (recognized stock combination), near zero when both ingredients are stock but the combination is chance-level (potentially original combination), and negative when stock ingredients are anti-correlated.

The composite functional form is the one design choice the Methods Appendix §7.3 leaves open. The three component columns (`z_ensemble_i`, `z_ensemble_j`, `lift`) are saved separately so any alternative composite can be recomputed downstream without re-running the notebook.

## Inputs

- `v4_rebuild/03b_classification/stockiness_long_classified.parquet` — 03b's enriched stockiness table, the canonical source for ensemble z-scores and the `is_narrative_element` flag.
- `v4_rebuild/02_elements/element_index.parquet` — 02's long-format film-element index. Used to rebuild per-film membership at `(element, element_type)` granularity because 03b's table aggregates over films within the genre, losing the per-film co-occurrence structure this notebook needs.
- `v4_rebuild/01c_union/movies_union.parquet` — 01c's per-film table. Used to map each film to its canonical genres.

## Outputs

- `v4_rebuild/03c_network/element_centrality_by_genre.parquet` — one row per `(element, element_type, genre)`. Includes degree, eigenvector, PageRank, neighbor count, films-in-genre count, classification, and the eigenvector solver status (weighted-numpy / weighted-power / unweighted fallback).
- `v4_rebuild/03c_network/combination_stockiness.parquet` — one row per `(element_i, element_type_i, element_j, element_type_j, genre)` surviving the five-film joint floor. Columns: joint counts and frequencies (genre and corpus), within-genre PMI signed and positive, corpus lift and log-lift, pair-level posterior, per-element ensemble z-scores in this genre, composite combination-stockiness.
- `v4_rebuild/03c_network/genre_networks.pkl` — dict mapping each canonical genre to its `networkx.Graph`. Edge weights are within-genre positive PMI.
- `v4_rebuild/03c_network/network_report.txt` — per-genre network size, top-15 elements by eigenvector centrality, top-15 stock combinations (highest combination-stockiness), bottom-15 stock×stock combinations (lowest combination-stockiness; the originality candidates among co-occurring pairs).

## Cost and runtime

No API spend. Pure pandas, scipy.sparse, NetworkX. Expected runtime 5–15 minutes across all 17 genres on a standard Colab CPU runtime, dominated by the per-genre sparse matrix multiplications and the eigenvector centrality solves on Drama and Comedy (the two largest genres).

In [1]:
# ----------------------------------------------------------------------
# Setup and configuration
# ----------------------------------------------------------------------
import os
import pickle
from pathlib import Path

import numpy as np
import pandas as pd
import scipy.sparse as sp
import networkx as nx
from tqdm.auto import tqdm

# --- Google Drive -----------------------------------------------------
from google.colab import drive
drive.mount('/content/drive')

# --- Paths ------------------------------------------------------------
# PROJECT_ROOT mirrors the convention in every other notebook so an output
# directory move can be done with a single edit.
ROOT = Path('/content/drive/MyDrive/DGK Lab/Reasoning Lab/'
            'DGK Lab - Scene A Faire Computational')
V4   = ROOT / 'Analysis/v4_rebuild'

PATH_ENRICHED  = V4 / '03b_classification' / 'stockiness_long_classified.parquet'
PATH_ELEM_IDX  = V4 / '02_elements'        / 'element_index.parquet'
PATH_MOVIES    = V4 / '01c_union'          / 'movies_union.parquet'

OUT_DIR        = V4 / '03c_network'
OUT_DIR.mkdir(parents=True, exist_ok=True)

PATH_CENTRALITY    = OUT_DIR / 'element_centrality_by_genre.parquet'
PATH_COMBINATIONS  = OUT_DIR / 'combination_stockiness.parquet'
PATH_NETWORKS      = OUT_DIR / 'genre_networks.pkl'
PATH_REPORT        = OUT_DIR / 'network_report.txt'

# --- Run configuration ------------------------------------------------
# Both filters match the corpus-wide convention in 03. An element needs at
# least 5 films in the genre to qualify as a node in that genre\'s graph;
# a pair needs at least 5 films of joint co-occurrence in the genre to
# qualify as an edge. The conjunction protects the network from the same
# hapax-driven instability the 10/5 filter protects 03 against.
MIN_FILMS_PER_NODE_IN_GENRE = 5
COOCCUR_FLOOR              = 5

# Dirichlet concentration for the pair-level posterior P(g | e_i, e_j).
# Matches the per-element posterior smoothing in 03 (alpha = 1, add-one),
# so the pair posterior is on the same footing as the smoothed per-element
# posterior that is in the ensemble.
ALPHA_SMOOTH = 1.0

# PageRank damping. Brin & Page 1998 standard; NetworkX default; the value
# every applied PageRank analysis uses unless there is a specific reason.
PAGERANK_DAMPING = 0.85

# Eigenvector centrality solver settings. We try the numpy-backed solver
# (scipy.sparse.linalg.eigs) first because it is more reliable on weighted
# graphs with disconnected pieces; fall back to power iteration if numpy
# fails. The power-iteration tolerances are slightly generous so very
# small genres do not spin on the limit unnecessarily.
EIGENVECTOR_POWER_MAX_ITER = 2000
EIGENVECTOR_POWER_TOL      = 1e-7

# Canonical genre list, identical to 03\'s. Hardcoded here so the notebook
# is self-contained and re-orderable.
CANONICAL_GENRES = [
    'Action', 'Adventure', 'Animation', 'Comedy', 'Crime',
    'Drama', 'Family', 'Fantasy', 'History', 'Horror',
    'Music', 'Mystery', 'Romance', 'Science Fiction',
    'Thriller', 'War', 'Western',
]
N_GENRES = len(CANONICAL_GENRES)

print('Setup complete.')
print(f'  output dir : {OUT_DIR}')
print(f'  node floor : {MIN_FILMS_PER_NODE_IN_GENRE} films per genre')
print(f'  edge floor : {COOCCUR_FLOOR} joint films per genre')
print(f'  alpha      : {ALPHA_SMOOTH}')
print(f'  damping    : {PAGERANK_DAMPING}')


Mounted at /content/drive
Setup complete.
  output dir : /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/03c_network
  node floor : 5 films per genre
  edge floor : 5 joint films per genre
  alpha      : 1.0
  damping    : 0.85


In [2]:
# ----------------------------------------------------------------------
# Load inputs
# ----------------------------------------------------------------------
# Three parquets:
#   - stockiness_long_classified : 03b\'s enriched table, one row per
#     (element, element_type, genre) cell. Carries 03\'s stockiness
#     measures (Primary, TF-IDF, Dunning, Posteriors, Ensemble) plus
#     03b\'s element_taxonomy and is_narrative_element flag.
#   - element_index              : 02\'s long-format film-element index.
#     One row per element occurrence in a film. Needed to rebuild per-film
#     membership at (element, element_type) granularity: 03b\'s table
#     aggregates within (element, type, genre), so it cannot tell us
#     which two elements co-appeared in which specific film.
#   - movies_union               : 01c\'s per-film table. The canonical
#     genres list per film is what we use to partition films by genre.
# ----------------------------------------------------------------------

print('Loading stockiness_long_classified...')
stk_cls = pd.read_parquet(PATH_ENRICHED)
print(f'  {len(stk_cls):,} rows × {stk_cls.shape[1]} cols')
assert stk_cls['is_narrative_element'].dtype == bool, (
    'is_narrative_element must be a true boolean; downstream filters '
    'assume that. Coerce in 03b if this fails.'
)

print('Loading element_index...')
ei = pd.read_parquet(PATH_ELEM_IDX)
print(f'  {len(ei):,} element rows across {ei["union_id"].nunique():,} films')

print('Loading movies_union...')
mv = pd.read_parquet(PATH_MOVIES)
# parquet round-trip turns list columns into numpy arrays; coerce back so
# downstream `genre in gs` checks work uniformly across all rows.
mv['genres_canonical'] = mv['genres_canonical'].apply(
    lambda gs: list(gs) if hasattr(gs, '__len__') and not isinstance(gs, str) else []
)
print(f'  {len(mv):,} films')

# The universe of nodes for every per-genre graph: every (element,
# element_type) pair that survived 03\'s 10/5 filter AND was tagged
# narrative by 03b. Sort and dedupe so the integer indexing downstream
# is deterministic across runs.
narrative_pairs = (stk_cls[stk_cls['is_narrative_element']]
                     [['element', 'element_type']]
                     .drop_duplicates()
                     .sort_values(['element', 'element_type'])
                     .reset_index(drop=True))
print(f'\nNarrative pairs (universe of network nodes): {len(narrative_pairs):,}')
print('  breakdown by element_type:')
print(narrative_pairs['element_type'].value_counts().to_string())


Loading stockiness_long_classified...
  456,159 rows × 38 cols
Loading element_index...
  5,658,714 element rows across 45,956 films
Loading movies_union...
  45,958 films

Narrative pairs (universe of network nodes): 28,066
  breakdown by element_type:
element_type
chunk                 14409
noun                   9291
verb                   3350
entity_NORP             385
entity_ORG              379
entity_PRODUCT           86
entity_GPE               49
entity_LOC               39
entity_WORK_OF_ART       31
entity_FAC               27
entity_EVENT             20


In [3]:
# ----------------------------------------------------------------------
# Build the corpus-wide film × narrative-pair membership matrix M_corpus.
#
# M_corpus[f, p] = 1 if film f contains at least one occurrence of the
# narrative pair p; 0 otherwise.
#
# Why this matrix is the right abstraction:
#   - Sparse storage. Films contain ~30-80 of the ~26K narrative pairs on
#     average, so the matrix is < 1% dense.
#   - Co-occurrence = M.T @ M. The sparse matmul gives every pair\'s
#     joint-film count in one shot, both corpus-wide (this cell\'s setup)
#     and per-genre (the per-genre loop below, with M sliced to F_g).
#   - Genre slicing is a row index operation, which is O(|F_g|) and does
#     not require recomputing per-film element lists for each genre.
#
# Steps:
#   (a) Restrict element_index to narrative pairs (drops the ~14.5K
#       elements 03b filtered as proper names, generic terms, etc.).
#   (b) Deduplicate to one row per (film, element, element_type). The
#       original index has one row per occurrence; a film containing
#       "sheriff" three times contributes three rows. For membership we
#       only need the binary fact.
#   (c) Build integer indices for films and pairs and assemble the CSR.
# ----------------------------------------------------------------------

# --- (a) Restrict element_index to narrative pairs --------------------
# inner merge on (element, element_type) keeps exactly the rows whose
# pair is in the narrative set.
ei_narr = ei.merge(narrative_pairs, on=['element', 'element_type'], how='inner')
print(f'After restricting to narrative pairs: {len(ei_narr):,} rows '
      f'(down from {len(ei):,} = {100 * len(ei_narr) / len(ei):.1f}%)')

# --- (b) Deduplicate to binary film-pair membership -------------------
ei_narr = ei_narr.drop_duplicates(
    ['union_id', 'element', 'element_type']
).reset_index(drop=True)
print(f'After deduplication to (film, pair) tuples: {len(ei_narr):,}')

# --- (c) Integer indices and sparse assembly ---------------------------
# Films: use the full union film list (45,958) so per-genre operations
# downstream are pure row-slices on a fixed coordinate space.
films_list = sorted(mv['union_id'].unique())
film_to_idx = {f: i for i, f in enumerate(films_list)}
N_films = len(films_list)
print(f'Film coordinate space: {N_films:,} films')

# Pairs: use the narrative_pairs frame so column index is constant across
# all genres. pair_to_idx maps a (element, element_type) tuple to its
# column position; pair_tuples is the inverse list.
pair_tuples = list(zip(narrative_pairs['element'], narrative_pairs['element_type']))
pair_to_idx = {p: i for i, p in enumerate(pair_tuples)}
N_pairs = len(pair_tuples)
print(f'Pair coordinate space: {N_pairs:,} narrative pairs')

# Vectorized index lookups for the sparse-matrix triplets.
# Building the (element, element_type) tuple column once is faster than
# repeated .apply(); .map against a dict is fast.
rows = ei_narr['union_id'].map(film_to_idx).values
cols = pd.Series(list(zip(ei_narr['element'], ei_narr['element_type']))).map(pair_to_idx).values

# Defensive: every row should map to a valid film and pair after the
# merge in (a). The check below catches any silent mismatch (e.g. if the
# narrative_pairs frame and element_index diverged).
valid = ~pd.isna(rows) & ~pd.isna(cols)
if not valid.all():
    print(f'WARNING: {(~valid).sum()} unmapped rows dropped before matrix build')
    rows = rows[valid]
    cols = cols[valid]

data = np.ones(len(rows), dtype=np.float32)
M_corpus = sp.csr_matrix(
    (data, (rows.astype(np.int32), cols.astype(np.int32))),
    shape=(N_films, N_pairs),
)
# Deduplication in step (b) makes each (film, pair) cell at most 1 to
# begin with, but the CSR constructor sums duplicate triplets, so we
# additionally clip to ensure binary membership.
M_corpus.data = np.minimum(M_corpus.data, 1.0)

print(f'\nM_corpus: {M_corpus.shape}, nnz={M_corpus.nnz:,}, '
      f'density={M_corpus.nnz / (N_films * N_pairs):.4%}')

# Sanity: row sums = number of narrative pairs per film.
per_film_counts = np.asarray(M_corpus.sum(axis=1)).ravel()
print(f'\nNarrative pairs per film:')
print(f'  median  : {int(np.median(per_film_counts))}')
print(f'  mean    : {per_film_counts.mean():.1f}')
print(f'  max     : {int(per_film_counts.max())}')
print(f'  films with 0 narrative pairs: {int((per_film_counts == 0).sum())}')


After restricting to narrative pairs: 3,278,222 rows (down from 5,658,714 = 57.9%)
After deduplication to (film, pair) tuples: 2,485,051
Film coordinate space: 45,958 films
Pair coordinate space: 28,066 narrative pairs

M_corpus: (45958, 28066), nnz=2,485,051, density=0.1927%

Narrative pairs per film:
  median  : 39
  mean    : 54.1
  max     : 555
  films with 0 narrative pairs: 5


In [4]:
# ----------------------------------------------------------------------
# Compute the corpus-wide co-occurrence matrix C_corpus = M_corpus.T @ M_corpus.
#
# Properties:
#   - Symmetric, shape (N_pairs, N_pairs).
#   - Diagonal C[p, p] = corpus-wide film count for pair p. This equals
#     total_film_count for the (element, element_type) row in stockiness_long
#     up to the multi-label explode arithmetic (which only matters for the
#     instance count, not the film count).
#   - Off-diagonal C[i, j] = number of unique films containing both pair i
#     AND pair j. This is the corpus-wide joint film count, the denominator
#     for the corpus PMI and the pair-level posterior\'s denominator.
#
# This is the single most memory-intensive step of the notebook. With
# N_pairs ≈ 26K and the density we have, expect somewhere in the
# 100-400 MB range of sparse storage.
# ----------------------------------------------------------------------

print('Computing corpus-wide co-occurrence C_corpus = M_corpus.T @ M_corpus ...')
C_corpus = (M_corpus.T @ M_corpus).tocsr()
print(f'C_corpus: {C_corpus.shape}, nnz={C_corpus.nnz:,}, '
      f'density={C_corpus.nnz / (N_pairs * N_pairs):.4%}')

# Store counts as int32; the underlying scipy matmul returns float32 and
# we want clean integer counts in the downstream lookups.
C_corpus.data = C_corpus.data.astype(np.int32)

# Diagonal = corpus film count per pair. Needed for the lift denominator
# and the pair-level posterior denominator.
pair_corpus_film_count = np.asarray(C_corpus.diagonal()).astype(np.int64)
print(f'\nPer-pair corpus film count:')
print(f'  min     : {int(pair_corpus_film_count.min())}  '
      f'(should be ≥5 — matches the 03 film filter)')
print(f'  median  : {int(np.median(pair_corpus_film_count))}')
print(f'  max     : {int(pair_corpus_film_count.max())}')


Computing corpus-wide co-occurrence C_corpus = M_corpus.T @ M_corpus ...
C_corpus: (28066, 28066), nnz=50,331,076, density=6.3896%

Per-pair corpus film count:
  min     : 5  (should be ≥5 — matches the 03 film filter)
  median  : 12
  max     : 14699


In [5]:
# ----------------------------------------------------------------------
# Per-genre network construction.
#
# For each canonical genre g:
#   (a) F_g = films tagged with g.
#   (b) M_g = M_corpus sliced to those rows. Shape (|F_g|, N_pairs).
#   (c) per-pair film counts in this genre = column sums of M_g.
#       Filter to active nodes (≥5 films in g).
#   (d) Restrict M_g to active columns: M_g_active.
#   (e) C_g = M_g_active.T @ M_g_active. Diagonal = local film counts;
#       off-diagonal = within-genre joint film counts.
#   (f) Extract pairs with joint_count ≥ COOCCUR_FLOOR (the edges).
#   (g) Within-genre PMI per edge. Positive PMI is the edge weight.
#   (h) Build a NetworkX graph and compute three centralities.
#   (i) Compute combination-stockiness per surviving pair: joint counts
#       (genre and corpus), pair-level posterior P(g | i, j), lift, the
#       two ensemble z-scores, the composite score.
#
# Two accumulators collect per-node and per-pair records across all
# genres; they are concatenated into long-format DataFrames after the
# loop and saved as parquets.
# ----------------------------------------------------------------------

# Genre prior P_prior(g), film-based. Matches the 03 posterior_smoothed
# prior so the pair-level posterior uses the same denominator
# decomposition (genre share of corpus film-genre pairs after the
# multi-label explode).
films_per_genre = {}
for g in CANONICAL_GENRES:
    films_per_genre[g] = set(
        mv[mv['genres_canonical'].apply(lambda gs: g in gs)]['union_id']
    )
total_film_genre_pairs = sum(len(s) for s in films_per_genre.values())
genre_prior_films = {
    g: len(s) / total_film_genre_pairs for g, s in films_per_genre.items()
}
assert abs(sum(genre_prior_films.values()) - 1.0) < 1e-9, 'prior must sum to 1'

# Per-element ensemble z-score lookup, indexed by (element, element_type, genre).
# Built once so the per-edge composite computation is O(1) per edge.
z_lookup = (stk_cls.set_index(['element', 'element_type', 'genre'])['ensemble']
            .to_dict())

# Per-element classification lookup, indexed by (element, element_type).
# Carried through to centrality and combination tables so they are
# interpretable without a downstream merge to 03b.
cls_lookup = (stk_cls[['element', 'element_type', 'element_taxonomy']]
              .drop_duplicates()
              .set_index(['element', 'element_type'])['element_taxonomy']
              .to_dict())

centrality_rows = []
combination_frames = []        # accumulate per-genre DataFrames, concat at end
genre_networks    = {}


def _solve_eigenvector(G):
    """Return (centrality_dict, status_string).

    Try the numpy-backed solver first (uses scipy.sparse.linalg.eigs and is
    more reliable on weighted graphs than power iteration). Fall back to
    weighted power iteration on numpy failure, then to unweighted power
    iteration as last resort. status_string lets the report flag any
    fallback.
    """
    try:
        return nx.eigenvector_centrality_numpy(G, weight='weight'), 'weighted_numpy'
    except Exception:
        pass
    try:
        eig = nx.eigenvector_centrality(
            G, weight='weight',
            max_iter=EIGENVECTOR_POWER_MAX_ITER,
            tol=EIGENVECTOR_POWER_TOL,
        )
        return eig, 'weighted_power_fallback'
    except nx.PowerIterationFailedConvergence:
        pass
    eig = nx.eigenvector_centrality(
        G,
        max_iter=EIGENVECTOR_POWER_MAX_ITER,
        tol=EIGENVECTOR_POWER_TOL,
    )
    return eig, 'unweighted_fallback'


for genre in tqdm(CANONICAL_GENRES, desc='per-genre'):
    # --- (a) F_g: films tagged with this genre -------------------------
    f_g = films_per_genre[genre]
    f_g_idx = np.array([film_to_idx[fid] for fid in f_g], dtype=np.int32)
    if len(f_g_idx) == 0:
        print(f'  WARNING: no films for {genre}; skipping.')
        continue
    N_g = len(f_g_idx)

    # --- (b) Slice M_corpus to F_g rows -------------------------------
    M_g = M_corpus[f_g_idx, :]

    # --- (c) Per-pair film counts in this genre -----------------------
    # column sum of M_g = films in g containing each pair.
    pair_film_count_g = np.asarray(M_g.sum(axis=0)).ravel().astype(np.int64)

    # Active nodes: pairs with at least MIN_FILMS_PER_NODE_IN_GENRE films
    # in the genre. We work with positions into pair_tuples / narrative_pairs.
    active_node_pos = np.where(
        pair_film_count_g >= MIN_FILMS_PER_NODE_IN_GENRE
    )[0]
    n_nodes_g = len(active_node_pos)

    if n_nodes_g < 2:
        print(f'  WARNING: only {n_nodes_g} active nodes in {genre}; skipping.')
        continue

    # --- (d) Restrict M_g to active columns ----------------------------
    M_g_active = M_g[:, active_node_pos]
    local_film_counts = pair_film_count_g[active_node_pos]

    # --- (e) Within-genre co-occurrence matrix -------------------------
    C_g = (M_g_active.T @ M_g_active).tocsr()

    # --- (f) Extract surviving edges -----------------------------------
    # COO triplets let us filter vectorized. Keep upper triangle only
    # (row < col) and apply the joint-film floor.
    C_g_coo = C_g.tocoo()
    keep = (C_g_coo.row < C_g_coo.col) & (C_g_coo.data >= COOCCUR_FLOOR)
    edge_local_i = C_g_coo.row[keep]
    edge_local_j = C_g_coo.col[keep]
    edge_joint   = C_g_coo.data[keep].astype(np.int64)

    if len(edge_joint) == 0:
        print(f'  WARNING: no edges survive the floor for {genre}; '
              f'recording isolated centrality rows only.')
        # Still emit zero-degree rows for the active nodes so the
        # centrality table covers every (element, type, genre) the
        # downstream notebooks might query.
        for pos in active_node_pos:
            node = pair_tuples[pos]
            z_e = z_lookup.get((node[0], node[1], genre), np.nan)
            centrality_rows.append({
                'genre': genre,
                'element': node[0],
                'element_type': node[1],
                'element_taxonomy': cls_lookup.get(node, 'unknown'),
                'n_films_in_genre': int(pair_film_count_g[pos]),
                'n_neighbors': 0,
                'degree': 0.0,
                'eigenvector': 0.0,
                'pagerank': 1.0 / n_nodes_g,
                'z_ensemble_in_genre': z_e,
                'eigenvector_status': 'no_edges',
            })
        genre_networks[genre] = nx.Graph()
        for pos in active_node_pos:
            genre_networks[genre].add_node(pair_tuples[pos])
        continue

    # Map local indices back to global pair indices.
    edge_global_i = active_node_pos[edge_local_i]
    edge_global_j = active_node_pos[edge_local_j]

    # --- (g) Within-genre PMI -------------------------------------------
    # PMI signed = log(P(i,j | g) / (P(i | g) * P(j | g))).
    # P(i,j | g) = joint_count / N_g
    # P(k   | g) = local_film_counts[k] / N_g
    # All quantities are positive by construction (counts ≥ 1 and N_g ≥ 1).
    p_eg  = edge_joint                 / N_g
    p_e_i = local_film_counts[edge_local_i] / N_g
    p_e_j = local_film_counts[edge_local_j] / N_g
    pmi_signed   = np.log(p_eg / (p_e_i * p_e_j))
    pmi_positive = np.clip(pmi_signed, 0.0, None)

    # --- (h) NetworkX graph + centrality --------------------------------
    G = nx.Graph()
    for pos in active_node_pos:
        G.add_node(pair_tuples[pos])
    edges = [
        (pair_tuples[edge_global_i[k]],
         pair_tuples[edge_global_j[k]],
         float(pmi_positive[k]))
        for k in range(len(edge_joint))
    ]
    G.add_weighted_edges_from(edges)
    genre_networks[genre] = G

    deg = nx.degree_centrality(G)
    eig, eig_status = _solve_eigenvector(G)
    pr  = nx.pagerank(G, alpha=PAGERANK_DAMPING, weight='weight')

    for pos in active_node_pos:
        node = pair_tuples[pos]
        z_e = z_lookup.get((node[0], node[1], genre), np.nan)
        centrality_rows.append({
            'genre': genre,
            'element': node[0],
            'element_type': node[1],
            'element_taxonomy': cls_lookup.get(node, 'unknown'),
            'n_films_in_genre': int(pair_film_count_g[pos]),
            'n_neighbors': G.degree(node),
            'degree': deg[node],
            'eigenvector': eig.get(node, 0.0),
            'pagerank': pr[node],
            'z_ensemble_in_genre': z_e,
            'eigenvector_status': eig_status,
        })

    # --- (i) Combination stockiness per surviving pair ------------------
    # Corpus joint counts (lookup into C_corpus by edge endpoints).
    joint_count_corpus = np.asarray(
        C_corpus[edge_global_i, edge_global_j]
    ).ravel().astype(np.int64)

    # Corpus marginals.
    corpus_marg_i = pair_corpus_film_count[edge_global_i]
    corpus_marg_j = pair_corpus_film_count[edge_global_j]

    # Lift, computed from counts so the N_films factor is explicit.
    # lift > 0 unless joint_count_corpus == 0, which should not occur:
    # a pair surviving the within-genre floor of 5 obviously co-occurs
    # corpus-wide at least 5 times. The np.where guard is purely
    # defensive in case future filter changes make zero possible.
    lift = (joint_count_corpus * N_films) / (corpus_marg_i * corpus_marg_j)
    safe_lift = np.where(lift > 0, lift, 1e-12)
    log_lift = np.log(safe_lift)

    # Pair-level posterior P(g | e_i, e_j), Dirichlet-smoothed.
    p_prior_g = genre_prior_films[genre]
    pair_posterior_smoothed = (
        (edge_joint + ALPHA_SMOOTH * p_prior_g)
        / (joint_count_corpus + ALPHA_SMOOTH)
    )

    joint_freq_g = edge_joint / N_g

    # Per-element ensemble z-scores for THIS genre. Build a fast lookup
    # by active node position so the per-edge dereferencing is constant
    # time. Missing entries (should not happen, but defensive) are NaN.
    active_node_z = np.empty(len(active_node_pos), dtype=np.float64)
    for k, pos in enumerate(active_node_pos):
        node = pair_tuples[pos]
        active_node_z[k] = z_lookup.get((node[0], node[1], genre), np.nan)
    z_i = active_node_z[edge_local_i]
    z_j = active_node_z[edge_local_j]

    composite = 0.5 * (z_i + z_j) * log_lift

    # Emit one row per surviving pair using vectorized DataFrame
    # construction (per-row dict appends are slow at hundreds of K rows).
    el_i = [pair_tuples[gi][0] for gi in edge_global_i]
    et_i = [pair_tuples[gi][1] for gi in edge_global_i]
    el_j = [pair_tuples[gj][0] for gj in edge_global_j]
    et_j = [pair_tuples[gj][1] for gj in edge_global_j]
    tax_i = [cls_lookup.get(pair_tuples[gi], 'unknown') for gi in edge_global_i]
    tax_j = [cls_lookup.get(pair_tuples[gj], 'unknown') for gj in edge_global_j]

    combination_frames.append(pd.DataFrame({
        'genre':            genre,
        'element_i':        el_i,
        'element_type_i':   et_i,
        'taxonomy_i':       tax_i,
        'element_j':        el_j,
        'element_type_j':   et_j,
        'taxonomy_j':       tax_j,
        'joint_count_g':    edge_joint,
        'joint_freq_g':     joint_freq_g,
        'joint_count_corpus': joint_count_corpus,
        'pmi_signed':       pmi_signed,
        'pmi_positive':     pmi_positive,
        'lift':             lift,
        'log_lift':         log_lift,
        'posterior_smoothed': pair_posterior_smoothed,
        'z_ensemble_i':     z_i,
        'z_ensemble_j':     z_j,
        'combination_stock': composite,
    }))

centrality_df  = pd.DataFrame(centrality_rows)
combination_df = (pd.concat(combination_frames, ignore_index=True)
                  if combination_frames else
                  pd.DataFrame(columns=[
                      'genre', 'element_i', 'element_type_i', 'taxonomy_i',
                      'element_j', 'element_type_j', 'taxonomy_j',
                      'joint_count_g', 'joint_freq_g', 'joint_count_corpus',
                      'pmi_signed', 'pmi_positive', 'lift', 'log_lift',
                      'posterior_smoothed', 'z_ensemble_i', 'z_ensemble_j',
                      'combination_stock',
                  ]))

print()
print(f'centrality_df  : {len(centrality_df):,} rows (one per node × genre)')
print(f'combination_df : {len(combination_df):,} rows (one per surviving pair × genre)')
print(f'genre_networks : {len(genre_networks)} graphs')


per-genre:   0%|          | 0/17 [00:00<?, ?it/s]


centrality_df  : 120,604 rows (one per node × genre)
combination_df : 9,224,996 rows (one per surviving pair × genre)
genre_networks : 17 graphs


In [6]:
# ----------------------------------------------------------------------
# Save outputs.
# ----------------------------------------------------------------------
centrality_df.to_parquet(PATH_CENTRALITY, index=False)
print(f'Saved: {PATH_CENTRALITY}  '
      f'({os.path.getsize(PATH_CENTRALITY)/1e6:.2f} MB, {len(centrality_df):,} rows)')

combination_df.to_parquet(PATH_COMBINATIONS, index=False)
print(f'Saved: {PATH_COMBINATIONS}  '
      f'({os.path.getsize(PATH_COMBINATIONS)/1e6:.2f} MB, {len(combination_df):,} rows)')

with open(PATH_NETWORKS, 'wb') as f:
    pickle.dump(genre_networks, f, protocol=pickle.HIGHEST_PROTOCOL)
print(f'Saved: {PATH_NETWORKS}  '
      f'({os.path.getsize(PATH_NETWORKS)/1e6:.2f} MB, {len(genre_networks)} graphs)')


Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/03c_network/element_centrality_by_genre.parquet  (4.45 MB, 120,604 rows)
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/03c_network/combination_stockiness.parquet  (440.82 MB, 9,224,996 rows)
Saved: /content/drive/MyDrive/DGK Lab/Reasoning Lab/DGK Lab - Scene A Faire Computational/Analysis/v4_rebuild/03c_network/genre_networks.pkl  (297.01 MB, 17 graphs)


In [7]:
# ----------------------------------------------------------------------
# Diagnostic report.
#
# Layout, per genre:
#   - Network size: nodes, edges, density, eigenvector solver status.
#   - Top 15 elements by eigenvector centrality (the load-bearing
#     conventions: what is constitutive of the genre\'s vocabulary).
#   - Top 15 stock combinations: highest composite combination-stockiness.
#     These are conventional pairings the doctrine would treat as part of
#     the genre\'s stock vocabulary at the combination level.
#   - Bottom 15 stock×stock combinations: lowest composite among pairs
#     where BOTH elements have ensemble_z >= 1 in the genre. These are
#     the "originality candidates" — stock ingredients combining at
#     chance rate or rarely, which is the configuration most likely to
#     retain originality even though the ingredients do not.
# ----------------------------------------------------------------------
TOP_K_REPORT     = 15
Z_STOCK_THRESHOLD = 1.0  # both ensemble_z >= this to count as a "stock×stock" pair

report_lines = []
def out(line=''):
    print(line)
    report_lines.append(line)

out('=' * 70)
out('NOTEBOOK 03c: CO-OCCURRENCE NETWORK & COMBINATION ANALYSIS REPORT')
out('=' * 70)
out(f'narrative pair universe (node universe) : {N_pairs:,}')
out(f'films in corpus                         : {N_films:,}')
out(f'node floor (≥ films per genre)          : {MIN_FILMS_PER_NODE_IN_GENRE}')
out(f'edge floor (≥ joint films per genre)    : {COOCCUR_FLOOR}')
out(f'Dirichlet alpha (pair posterior)        : {ALPHA_SMOOTH}')
out(f'PageRank damping                        : {PAGERANK_DAMPING}')
out(f'Z stock threshold (originality cohort)  : {Z_STOCK_THRESHOLD}')
out('')

# --- Per-genre network size table -------------------------------------
out('PER-GENRE NETWORK SIZE')
out('-' * 70)
out(f'{"genre":<18} {"nodes":>7} {"edges":>9} {"density":>10}   eig_status')
for genre in CANONICAL_GENRES:
    if genre not in genre_networks:
        out(f'{genre:<18}  (no graph)')
        continue
    G = genre_networks[genre]
    n = G.number_of_nodes()
    m = G.number_of_edges()
    density = (2 * m) / (n * (n - 1)) if n > 1 else 0.0
    eig_sub = centrality_df.loc[centrality_df['genre']==genre, 'eigenvector_status']
    eig_status = eig_sub.iloc[0] if len(eig_sub) else '(no rows)'
    out(f'{genre:<18} {n:>7,} {m:>9,} {density:>10.4f}   {eig_status}')
out('')

# --- Top central elements per genre (two views) ------------------------
# View A: raw eigenvector centrality — the structural hubs of the graph,
#         which tend to be the most-connected nodes regardless of how
#         distinctively they belong to the genre. Documents the network
#         topology but does NOT by itself identify genre-specific
#         conventions.
# View B: filtered to z_ensemble_in_genre >= 1.0, then ranked by
#         eigenvector — the structural hubs AMONG the elements that
#         are also distinctively stock in this genre. This is the
#         load-bearing-conventions view the doctrinal argument wants:
#         constitutive of the genre AND structurally central within it.
for genre in CANONICAL_GENRES:
    if genre not in genre_networks:
        continue
    rows = (centrality_df[centrality_df['genre']==genre]
              .nlargest(TOP_K_REPORT, 'eigenvector'))
    out('=' * 70)
    out(f'TOP {TOP_K_REPORT} BY EIGENVECTOR CENTRALITY (structural hubs) — {genre}')
    out('=' * 70)
    for i, row in enumerate(rows.itertuples(), 1):
        z_str = (f'{row.z_ensemble_in_genre:.2f}'
                 if pd.notna(row.z_ensemble_in_genre) else ' NaN')
        out(f'  {i:>2}. {row.element:<28} [{row.element_type:<14}] '
            f'tax={row.element_taxonomy:<16} '
            f'eig={row.eigenvector:.4f}  pr={row.pagerank:.4f}  '
            f'z_ens={z_str}  films={row.n_films_in_genre}  '
            f'deg={row.n_neighbors}')
    out('')

# View B
for genre in CANONICAL_GENRES:
    if genre not in genre_networks:
        continue
    sub = centrality_df[
        (centrality_df['genre']==genre) &
        (centrality_df['z_ensemble_in_genre'] >= Z_STOCK_THRESHOLD)
    ]
    if len(sub) == 0:
        continue
    rows = sub.nlargest(TOP_K_REPORT, 'eigenvector')
    out('=' * 70)
    out(f'TOP {TOP_K_REPORT} LOAD-BEARING CONVENTIONS — {genre}')
    out(f'  (eigenvector centrality among z_ensemble_in_genre >= {Z_STOCK_THRESHOLD})')
    out('=' * 70)
    for i, row in enumerate(rows.itertuples(), 1):
        z_str = (f'{row.z_ensemble_in_genre:.2f}'
                 if pd.notna(row.z_ensemble_in_genre) else ' NaN')
        out(f'  {i:>2}. {row.element:<28} [{row.element_type:<14}] '
            f'tax={row.element_taxonomy:<16} '
            f'eig={row.eigenvector:.4f}  pr={row.pagerank:.4f}  '
            f'z_ens={z_str}  films={row.n_films_in_genre}  '
            f'deg={row.n_neighbors}')
    out('')

# --- Top stock combinations per genre ---------------------------------
for genre in CANONICAL_GENRES:
    pairs_g = combination_df[combination_df['genre']==genre]
    if len(pairs_g) == 0:
        continue
    rows = pairs_g.nlargest(TOP_K_REPORT, 'combination_stock')
    out('=' * 70)
    out(f'TOP {TOP_K_REPORT} STOCK COMBINATIONS (high combination_stock) — {genre}')
    out('=' * 70)
    for i, row in enumerate(rows.itertuples(), 1):
        out(f'  {i:>2}. {row.element_i} [{row.element_type_i}] + '
            f'{row.element_j} [{row.element_type_j}]')
        out(f'      CS={row.combination_stock:.2f}  lift={row.lift:.2f}  '
            f'joint_g={row.joint_count_g}  '
            f'P_s(g|i,j)={row.posterior_smoothed:.3f}  '
            f'z_i={row.z_ensemble_i:.2f}  z_j={row.z_ensemble_j:.2f}')
    out('')

# --- Originality candidates per genre ---------------------------------
# Both ingredients stock in the genre AND the lowest composite among
# those — pairs where stock × stock co-occurs less than expected.
for genre in CANONICAL_GENRES:
    pairs_g = combination_df[
        (combination_df['genre']==genre) &
        (combination_df['z_ensemble_i'] >= Z_STOCK_THRESHOLD) &
        (combination_df['z_ensemble_j'] >= Z_STOCK_THRESHOLD)
    ]
    if len(pairs_g) == 0:
        continue
    rows = pairs_g.nsmallest(TOP_K_REPORT, 'combination_stock')
    out('=' * 70)
    out(f'BOTTOM {TOP_K_REPORT} STOCK×STOCK COMBINATIONS (low combination_stock')
    out(f'  = potentially original within co-occurring pairs) — {genre}')
    out('=' * 70)
    for i, row in enumerate(rows.itertuples(), 1):
        out(f'  {i:>2}. {row.element_i} [{row.element_type_i}] + '
            f'{row.element_j} [{row.element_type_j}]')
        out(f'      CS={row.combination_stock:.2f}  lift={row.lift:.2f}  '
            f'joint_g={row.joint_count_g}  '
            f'z_i={row.z_ensemble_i:.2f}  z_j={row.z_ensemble_j:.2f}')
    out('')

PATH_REPORT.write_text('\n'.join(report_lines))
print(f'Report written to {PATH_REPORT}')


NOTEBOOK 03c: CO-OCCURRENCE NETWORK & COMBINATION ANALYSIS REPORT
narrative pair universe (node universe) : 28,066
films in corpus                         : 45,958
node floor (≥ films per genre)          : 5
edge floor (≥ joint films per genre)    : 5
Dirichlet alpha (pair posterior)        : 1.0
PageRank damping                        : 0.85
Z stock threshold (originality cohort)  : 1.0

PER-GENRE NETWORK SIZE
----------------------------------------------------------------------
genre                nodes     edges    density   eig_status
Action               9,608   913,617     0.0198   weighted_power_fallback
Adventure            8,737   802,105     0.0210   weighted_power_fallback
Animation            5,200   304,377     0.0225   weighted_power_fallback
Comedy              12,927 1,063,966     0.0127   weighted_power_fallback
Crime                6,943   524,240     0.0218   weighted_power_fallback
Drama               16,909 1,635,081     0.0114   weighted_power_fallback
Family   

## Outputs produced

- **`element_centrality_by_genre.parquet`** — one row per `(element, element_type, genre)`. Columns:
  - `element_taxonomy`, `n_films_in_genre`, `n_neighbors`
  - `degree`, `eigenvector`, `pagerank`
  - `z_ensemble_in_genre` — per-element ensemble z-score in this genre, carried through from 03 so the centrality table can be filtered to genre-distinctive elements without a downstream merge
  - `eigenvector_status` — `weighted_numpy` (the normal case), `weighted_power_fallback`, `unweighted_fallback`, or `no_edges` (the genre had no edges at all)

- **`combination_stockiness.parquet`** — one row per `(element_i, element_type_i, element_j, element_type_j, genre)` surviving the joint-film floor (5). Columns:
  - `joint_count_g`, `joint_freq_g`, `joint_count_corpus`
  - `pmi_signed`, `pmi_positive` (within-genre)
  - `lift`, `log_lift` (corpus-wide; `lift = exp(corpus_pmi_signed)`)
  - `posterior_smoothed` — Dirichlet-smoothed pair-level posterior P(g | e_i, e_j)
  - `z_ensemble_i`, `z_ensemble_j` — the per-element ensemble z-scores from 03 in this genre
  - `combination_stock` — composite, `½(z_i + z_j) · log(lift)`

- **`genre_networks.pkl`** — dict mapping each canonical genre to its NetworkX graph. Nodes are `(element, element_type)` tuples; edges carry `weight` = within-genre positive PMI. Useful for downstream visualization, community detection, k-core decomposition, and ego networks for case studies.

- **`network_report.txt`** — top-15 per genre under each of three views: eigenvector centrality (load-bearing conventions), highest combination-stockiness (stock combinations), lowest stock×stock combination-stockiness (originality candidates among co-occurring pairs).

## What the report shows, per genre

Four ranked views surface the doctrinal questions the framework is built to answer:

1. **Top 15 by eigenvector centrality (structural hubs).** Pure graph structure. These are the elements at the structural core of the genre's co-occurrence network. On most genres this view is dominated by generic stock elements that connect to nearly everything — useful as a topology diagnostic but not directly answering the doctrinal question.

2. **Top 15 load-bearing conventions.** Eigenvector centrality among elements with `z_ensemble_in_genre >= 1.0`. This is the view that operationalizes "constitutive of the genre's vocabulary": structurally central *and* distinctively belonging to the genre.

3. **Top 15 stock combinations.** Highest composite `combination_stock`. Pairs of elements where both ingredients are stock in the genre AND the pair co-occurs super-additively. These are the conventional pairings the doctrine would treat as stock at the combination level.

4. **Bottom 15 stock×stock combinations (originality candidates).** Lowest composite among pairs where both elements have `z_ensemble_in_genre >= 1`. Stock ingredients combining at chance rate or rarely, which is the configuration most likely to retain originality even though the individual ingredients do not. The Indiana Jones case in operationalized form: low joint frequency of stock ingredients suggests the combination, not the ingredients, carries the originality.

## What feeds where

- **Notebook 04 (plot surprisal)** — a per-film average centrality, weighted by occurrences, would extend the discriminant-validity story to combinations.
- **Notebook 06 (convergent validation)** — centrality rankings can be compared against expert-curated lists of genre conventions; combination-stockiness rankings against doctrinal commentary on stock combinations.
- **Notebook 07 (court case validation)** — the most direct consumer. Judicial opinions frequently name specific combinations as scènes à faire ("the trope of Nazis hunting for a magical artifact," "the haunted hospital with a dying patient"). Matching court-named combinations against the combination-stockiness table, then computing precision-at-K, is the load-bearing validation for the doctrinal claim that the framework operationalizes the doctrine itself.

## Known limitations

- **The five-film joint floor cuts the long tail.** Pairs of stock elements that NEVER co-occur (or co-occur fewer than 5 times in a genre) are absent from the combination table. The "originality candidates" view shows surprisingly-low-co-occurrence pairs that DID survive the floor, which is informative but not exhaustive. A complementary "originality candidates from the dark matter" view, listing pairs of stock elements with zero or near-zero in-genre co-occurrence, can be added if it proves doctrinally useful and the paper needs it.
- **Composite functional form is one choice.** `½(z_i + z_j) · log(lift)` is intuitive (high when both ingredients are stock AND lift > 1; near zero on stock×stock chance-level pairs; negative on anti-correlated pairs) but not uniquely identified by theory. The three component columns are saved separately so any alternative composite can be computed downstream without re-running.
- **Within-genre PMI uses within-genre marginals.** This is consistent with the per-genre framing of the rest of the pipeline and with Methods Appendix §7.1. A corpus-wide PMI is also implicit in the `lift` column (`lift = exp(corpus_PMI_signed)`) for cross-genre comparison.
- **Disconnected nodes are kept in the centrality table** with `degree=0`, `eigenvector~0`, `pagerank ≈ 1/n`. They are real members of the genre's narrative vocabulary that happen not to combine with anything else at the five-film floor. Treat them like any other stock element when reading the centrality table.
- **Genre subgraphs are independent.** A node can have very different centrality in different genres — `sheriff` in Western vs. `sheriff` in Crime, for instance. This is by design (the doctrine treats each genre's vocabulary as its own object) but means the centrality table is genuinely a long-format relational table, not a node-level summary.